# DIM_MUNICIPIO

Municípios do Brasil, com região de saúde, macrorregião e tipo de localidade
(capital, interior, outra UF). As fatos fazem o lookup pelo código IBGE de 6 dígitos
(`COD_IBGE_6`, chave natural única).

Códigos sem correspondência (ex.: `999999` = residência não informada, `270000` =
município ignorado em AL) vão para o membro desconhecido.

**Estratégia de carga:** incremental com `MERGE` (SCD tipo 1). Registros novos recebem a surrogate key da sequence; registros existentes só são atualizados se algum atributo mudou. Reexecutar a carga não duplica linhas nem troca as chaves.

**Membro desconhecido:** `SK = -1` ("NAO INFORMADO"), usado pelas fatos quando o código de origem não existe na dimensão.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import mesclar, executar, consultar

In [ ]:
MEMBRO_DESCONHECIDO = """
MERGE INTO DIM_MUNICIPIO d
USING (SELECT -1 AS SK FROM DUAL) s ON (d.SK_MUNICIPIO = s.SK)
WHEN NOT MATCHED THEN INSERT
    (SK_MUNICIPIO, COD_IBGE_6, COD_IBGE_7, NOME_MUNICIPIO, UF, REGIAO_SAUDE, MACRORREGIAO, TIPO_LOCALIDADE)
VALUES (-1, '-1', NULL, 'NAO INFORMADO', 'NI', 'NAO INFORMADO', 'NAO INFORMADO', 'NAO INFORMADO')
"""
executar(MEMBRO_DESCONHECIDO)

In [ ]:
SQL = """
MERGE INTO DIM_MUNICIPIO d
USING SILVER_MUNICIPIO s
ON (d.COD_IBGE_6 = s.COD_IBGE_6)
WHEN MATCHED THEN UPDATE SET
    d.COD_IBGE_7 = s.COD_IBGE_7, d.NOME_MUNICIPIO = s.NOME_MUNICIPIO, d.UF = s.UF,
    d.REGIAO_SAUDE = s.REGIAO_SAUDE, d.MACRORREGIAO = s.MACRORREGIAO,
    d.TIPO_LOCALIDADE = s.TIPO_LOCALIDADE
  WHERE DECODE(d.NOME_MUNICIPIO, s.NOME_MUNICIPIO, 0, 1) + DECODE(d.REGIAO_SAUDE, s.REGIAO_SAUDE, 0, 1)
      + DECODE(d.MACRORREGIAO, s.MACRORREGIAO, 0, 1) + DECODE(d.TIPO_LOCALIDADE, s.TIPO_LOCALIDADE, 0, 1)
      + DECODE(d.COD_IBGE_7, s.COD_IBGE_7, 0, 1) + DECODE(d.UF, s.UF, 0, 1) > 0
WHEN NOT MATCHED THEN INSERT
    (SK_MUNICIPIO, COD_IBGE_6, COD_IBGE_7, NOME_MUNICIPIO, UF, REGIAO_SAUDE, MACRORREGIAO, TIPO_LOCALIDADE)
VALUES
    (SEQ_DIM_MUNICIPIO.NEXTVAL, s.COD_IBGE_6, s.COD_IBGE_7, s.NOME_MUNICIPIO, s.UF,
     s.REGIAO_SAUDE, s.MACRORREGIAO, s.TIPO_LOCALIDADE)
"""

In [ ]:
mesclar("DIM_MUNICIPIO", SQL, origem="SILVER_MUNICIPIO")

In [ ]:
consultar("SELECT * FROM DIM_MUNICIPIO ORDER BY 1 FETCH FIRST 10 ROWS ONLY")